<a href="https://colab.research.google.com/github/saadhana192465019/Digital-Forensics-and-cyber-crime-Investigation--CSA6102/blob/main/Experiment_50_END_TO_END_INVESTIGATION_OF_A_SIMULATED_INSIDER_DATA_THEFT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ================================================================
# EXPERIMENT 12 - CAPSTONE
# END-TO-END INVESTIGATION OF A SIMULATED INSIDER DATA THEFT
# ================================================================

import hashlib
import os
import re
import tempfile

from collections import Counter
from datetime import datetime, timezone, timedelta


# ================================================================
# TIME ZONES
# ================================================================

IST = timezone(timedelta(hours=5, minutes=30))
UTC = timezone.utc


# ================================================================
# SIMULATED CASE LOG
# ================================================================

CASE_LOG = """\
2026-08-19T18:02:11Z auth user=r.kumar host=WS0142 result=success src=10.20.3.41
2026-08-19T18:40:03Z file user=r.kumar action=read path=/finance/clients_2026.xlsx size=2411520
2026-08-19T19:05:44Z file user=r.kumar action=read path=/finance/pricing_master.xlsx size=1180288
2026-08-19T22:14:07Z net user=r.kumar host=WS0142 dst=203.0.113.90 port=443 bytes=1204
2026-08-19T22:44:07Z net user=r.kumar host=WS0142 dst=203.0.113.90 port=443 bytes=1198
2026-08-19T23:14:07Z net user=r.kumar host=WS0142 dst=203.0.113.90 port=443 bytes=1211
2026-08-19T23:44:07Z net user=r.kumar host=WS0142 dst=203.0.113.90 port=443 bytes=1207
2026-08-20T00:12:55Z auth user=svc_backup host=FS01 result=success src=10.20.3.41
2026-08-20T00:16:20Z admin user=svc_backup action=create_account newuser=svc_helper host=DC01
2026-08-20T00:31:09Z web user=svc_backup url=https://files.example.net/upload bytes=41231872
2026-08-20T00:44:31Z audit user=svc_backup action=clear_security_log host=FS01
2026-08-20T09:15:02Z auth user=r.kumar host=WS0142 result=success src=10.20.3.41
"""


# ================================================================
# INDICATOR OF COMPROMISE PATTERNS
# ================================================================

IOC_PATTERNS = {

    "ipv4":
        r"\b(?:\d{1,3}\.){3}\d{1,3}\b",

    "domain":
        r"https?://([a-z0-9.-]+\.[a-z]{2,})",

    "user":
        r"user=([\w.\-]+)",

    "host":
        r"host=([\w.\-]+)"
}


# ================================================================
# PARSE LOG
# ================================================================

def parse(log_text=CASE_LOG):

    events = []

    for line in log_text.strip().split("\n"):

        match = re.match(
            r"^(\S+Z)\s+(\w+)\s+(.*)$",
            line
        )

        if not match:
            continue

        timestamp = datetime.strptime(
            match.group(1),
            "%Y-%m-%dT%H:%M:%SZ"
        ).replace(tzinfo=UTC)

        events.append(
            {
                "utc": timestamp,
                "ist": timestamp.astimezone(IST),
                "type": match.group(2),
                "detail": match.group(3),
                "raw": line
            }
        )

    # Sort chronologically
    events.sort(
        key=lambda event: event["utc"]
    )

    return events


# ================================================================
# EXTRACT INDICATORS OF COMPROMISE
# ================================================================

def extract_iocs(log_text=CASE_LOG):

    output = {}

    for name, pattern in IOC_PATTERNS.items():

        values = re.findall(
            pattern,
            log_text
        )

        output[name] = sorted(
            set(values)
        )

    return output


# ================================================================
# DETECT BEACONING
# ================================================================

def detect_beaconing(
    events,
    tolerance_s=30
):

    timestamps = [

        event["utc"]

        for event in events

        if event["type"] == "net"
    ]

    if len(timestamps) < 3:

        return {
            "detected": False,
            "intervals": [],
            "period_s": None
        }

    gaps = [

        int(
            (
                timestamps[i]
                - timestamps[i - 1]
            ).total_seconds()
        )

        for i in range(
            1,
            len(timestamps)
        )
    ]

    detected = (
        max(gaps) - min(gaps)
        <= tolerance_s
    )

    return {

        "detected": detected,

        "intervals": gaps,

        "period_s":
            gaps[0] if gaps else None
    }


# ================================================================
# DETECT LARGE DATA TRANSFER
# ================================================================

def detect_exfiltration(
    events,
    threshold_bytes=10_000_000
):

    hits = []

    for event in events:

        match = re.search(
            r"bytes=(\d+)",
            event["detail"]
        )

        if match:

            byte_count = int(
                match.group(1)
            )

            if byte_count >= threshold_bytes:

                hits.append(
                    {
                        "ist":
                            event["ist"].strftime(
                                "%d-%m-%Y %H:%M IST"
                            ),

                        "bytes":
                            byte_count,

                        "detail":
                            event["detail"]
                    }
                )

    return hits


# ================================================================
# DETECT ANTI-FORENSICS
# ================================================================

def detect_antiforensics(events):

    return [

        event

        for event in events

        if "clear_security_log"
        in event["detail"]
    ]


# ================================================================
# DETECT PRIVILEGE ABUSE
# ================================================================

def detect_privilege_abuse(events):

    return [

        event

        for event in events

        if "create_account"
        in event["detail"]
    ]


# ================================================================
# DETECT OUT-OF-HOURS ACTIVITY
# ================================================================

def out_of_hours(
    events,
    start_h=22,
    end_h=6
):

    """
    Activity outside 06:00-22:00 IST
    is treated as out-of-hours.
    """

    return [

        event

        for event in events

        if not (
            end_h
            <= event["ist"].hour
            < start_h
        )
    ]


# ================================================================
# INDICATIVE LEGAL MAPPING
# Educational mapping only
# ================================================================

LEGAL_MAP = [

    (
        "Unauthorised access / dishonest use of another account",
        "IT Act s.66 r/w s.43"
    ),

    (
        "Fraudulent use of another person's credentials",
        "IT Act s.66C"
    ),

    (
        "Disclosure of information in breach of lawful contract",
        "IT Act s.72A"
    ),

    (
        "Copying and exfiltrating employer data",
        "IT Act s.43(b) and s.66"
    ),

    (
        "Deletion or alteration of audit records",
        "IT Act s.65 / s.66"
    ),

    (
        "Production of the electronic record in evidence",
        "Indian Evidence Act s.65B(4) certificate required"
    )
]


# ================================================================
# SHA-256 FUNCTION
# ================================================================

def sha256_of(text):

    return hashlib.sha256(
        text.encode()
    ).hexdigest()


# ================================================================
# COMPLETE INVESTIGATION
# ================================================================

def investigate():

    events = parse()

    findings = {

        "events":
            len(events),

        "iocs":
            extract_iocs(),

        "beaconing":
            detect_beaconing(events),

        "exfiltration":
            detect_exfiltration(events),

        "anti_forensics":
            len(
                detect_antiforensics(events)
            ),

        "privilege_abuse":
            len(
                detect_privilege_abuse(events)
            ),

        "out_of_hours_events":
            len(
                out_of_hours(events)
            ),

        "evidence_sha256":
            sha256_of(CASE_LOG)
    }

    return events, findings


# ================================================================
# INVESTIGATION SUMMARY
# ================================================================

def summarise(events, findings):

    lines = [

        "CAPSTONE INVESTIGATION SUMMARY - "
        "CASE/CYB/2026/0417",

        "=" * 62,

        f"Events examined : "
        f"{findings['events']}",

        f"Evidence file SHA-256 : "
        f"{findings['evidence_sha256']}",

        f"Distinct users observed : "
        f"{', '.join(findings['iocs']['user'])}",

        f"External addresses : "
        f"{', '.join(findings['iocs']['ipv4'])}",

        f"External domains : "
        f"{', '.join(findings['iocs']['domain'])}",

        f"Beaconing detected : "
        f"{findings['beaconing']['detected']} "
        f"(interval "
        f"{findings['beaconing']['period_s']} s)",

        f"Large outbound transfers : "
        f"{len(findings['exfiltration'])}"
    ]

    for transfer in findings["exfiltration"]:

        lines.append(

            f" {transfer['ist']} "
            f"{transfer['bytes']:,} bytes"
        )

    lines.extend([

        f"Privileged account created : "
        f"{findings['privilege_abuse']}",

        f"Audit log cleared : "
        f"{findings['anti_forensics']}",

        f"Out-of-hours events : "
        f"{findings['out_of_hours_events']}",

        "",

        "INDICATIVE LEGAL PROVISIONS "
        "(educational mapping, not legal advice)"
    ])

    for description, section in LEGAL_MAP:

        lines.append(

            f"{description:<58}"
            f"{section}"
        )

    return "\n".join(lines)


# ================================================================
# TIMELINE DISPLAY
# ================================================================

def print_timeline(events):

    print("\n" + "=" * 78)
    print("CHRONOLOGICAL TIMELINE")
    print("=" * 78)

    for event in events:

        print(
            f"{event['ist'].strftime('%d-%m-%Y %H:%M:%S IST')} | "
            f"{event['type']:<6} | "
            f"{event['detail']}"
        )


# ================================================================
# TEST CASES
# ================================================================

def run_tests():

    events, findings = investigate()

    results = []

    # ------------------------------------------------------------
    # TC1
    # ------------------------------------------------------------

    results.append(
        (
            "TC1 all 12 log lines parsed",
            findings["events"] == 12
        )
    )

    # ------------------------------------------------------------
    # TC2
    # ------------------------------------------------------------

    results.append(
        (
            "TC2 timeline is chronologically sorted",

            all(
                events[i]["utc"]
                >= events[i - 1]["utc"]

                for i in range(
                    1,
                    len(events)
                )
            )
        )
    )

    # ------------------------------------------------------------
    # TC3
    # ------------------------------------------------------------

    results.append(
        (
            "TC3 external IP extracted as IoC",

            "203.0.113.90"
            in findings["iocs"]["ipv4"]
        )
    )

    # ------------------------------------------------------------
    # TC4
    # ------------------------------------------------------------

    results.append(
        (
            "TC4 exfiltration domain extracted",

            "files.example.net"
            in findings["iocs"]["domain"]
        )
    )

    # ------------------------------------------------------------
    # TC5
    # ------------------------------------------------------------

    results.append(
        (
            "TC5 both accounts identified",

            set(
                [
                    "r.kumar",
                    "svc_backup"
                ]
            )
            <=
            set(
                findings["iocs"]["user"]
            )
        )
    )

    # ------------------------------------------------------------
    # TC6
    # ------------------------------------------------------------

    results.append(
        (
            "TC6 30-minute beaconing detected",

            findings["beaconing"]["detected"]
            and
            findings["beaconing"]["period_s"]
            == 1800
        )
    )

    # ------------------------------------------------------------
    # TC7
    # ------------------------------------------------------------

    results.append(
        (
            "TC7 41 MB transfer flagged",

            len(
                findings["exfiltration"]
            ) == 1

            and

            findings["exfiltration"][0]["bytes"]
            == 41231872
        )
    )

    # ------------------------------------------------------------
    # TC8
    # ------------------------------------------------------------

    results.append(
        (
            "TC8 small transfers not flagged",

            all(
                item["bytes"]
                > 10_000_000

                for item
                in findings["exfiltration"]
            )
        )
    )

    # ------------------------------------------------------------
    # TC9
    # ------------------------------------------------------------

    results.append(
        (
            "TC9 privileged account creation detected",

            findings["privilege_abuse"]
            == 1
        )
    )

    # ------------------------------------------------------------
    # TC10
    # ------------------------------------------------------------

    results.append(
        (
            "TC10 audit-log clearing detected",

            findings["anti_forensics"]
            == 1
        )
    )

    # ------------------------------------------------------------
    # TC11
    # ------------------------------------------------------------

    results.append(
        (
            "TC11 out-of-hours activity detected",

            findings["out_of_hours_events"]
            >= 5
        )
    )

    # ------------------------------------------------------------
    # TC12
    # ------------------------------------------------------------

    results.append(
        (
            "TC12 evidence hash is 64 hex chars",

            len(
                findings["evidence_sha256"]
            ) == 64

            and

            re.fullmatch(
                r"[0-9a-f]{64}",
                findings["evidence_sha256"]
            ) is not None
        )
    )

    # ------------------------------------------------------------
    # TC13
    # Integrity test
    # ------------------------------------------------------------

    altered_log = CASE_LOG.replace(
        "41231872",
        "41231873"
    )

    results.append(
        (
            "TC13 single-character edit changes hash",

            sha256_of(altered_log)
            !=
            findings["evidence_sha256"]
        )
    )

    # ------------------------------------------------------------
    # TC14
    # Correct event sequence
    # ------------------------------------------------------------

    event_details = [
        event["detail"]
        for event in events
    ]

    create_index = next(
        i

        for i, detail
        in enumerate(event_details)

        if "create_account"
        in detail
    )

    upload_index = next(
        i

        for i, detail
        in enumerate(event_details)

        if "files.example.net"
        in detail
    )

    clear_index = next(
        i

        for i, detail
        in enumerate(event_details)

        if "clear_security_log"
        in detail
    )

    results.append(
        (
            "TC14 create < upload < clear sequence",

            create_index
            < upload_index
            < clear_index
        )
    )

    # ------------------------------------------------------------
    # TC15
    # ------------------------------------------------------------

    results.append(
        (
            "TC15 s.65B admissibility noted",

            any(
                "65B" in section
                for _, section
                in LEGAL_MAP
            )
        )
    )

    # ============================================================
    # PRINT RESULTS
    # ============================================================

    print("\n" + "=" * 78)
    print("TEST CASE RESULTS")
    print("=" * 78)

    for name, passed in results:

        status = (
            "PASS"
            if passed
            else "FAIL"
        )

        print(
            f"{name:<50} -> {status}"
        )

    passed_count = sum(
        1
        for _, passed
        in results
        if passed
    )

    print("=" * 78)

    print(
        f"RESULT: {passed_count}/{len(results)} "
        "test cases passed"
    )

    return passed_count == len(results)


# ================================================================
# RUN COMPLETE CAPSTONE
# ================================================================

print("=" * 78)
print("EXPERIMENT 12")
print("CAPSTONE: END-TO-END INVESTIGATION")
print("OF A SIMULATED INSIDER DATA THEFT")
print("=" * 78)


# ================================================================
# INVESTIGATION
# ================================================================

events, findings = investigate()


# ================================================================
# PRINT SUMMARY
# ================================================================

print("\n")
print(summarise(events, findings))


# ================================================================
# PRINT TIMELINE
# ================================================================

print_timeline(events)


# ================================================================
# PRINT IOC DETAILS
# ================================================================

print("\n" + "=" * 78)
print("INDICATORS OF COMPROMISE")
print("=" * 78)

for category, values in findings["iocs"].items():

    print(
        f"\n{category.upper()}:"
    )

    for value in values:

        print(
            f"  - {value}"
        )


# ================================================================
# PRINT BEACONING DETAILS
# ================================================================

print("\n" + "=" * 78)
print("BEACONING ANALYSIS")
print("=" * 78)

print(
    "Detected :",
    findings["beaconing"]["detected"]
)

print(
    "Intervals :",
    findings["beaconing"]["intervals"],
    "seconds"
)

print(
    "Periodic interval :",
    findings["beaconing"]["period_s"],
    "seconds"
)


# ================================================================
# PRINT EXFILTRATION
# ================================================================

print("\n" + "=" * 78)
print("EXFILTRATION ANALYSIS")
print("=" * 78)

if findings["exfiltration"]:

    for transfer in findings["exfiltration"]:

        print(
            f"Time : {transfer['ist']}"
        )

        print(
            f"Bytes transferred : "
            f"{transfer['bytes']:,}"
        )

else:

    print("No large outbound transfer detected.")


# ================================================================
# PRINT INTEGRITY
# ================================================================

print("\n" + "=" * 78)
print("EVIDENCE INTEGRITY")
print("=" * 78)

print(
    "SHA-256:",
    findings["evidence_sha256"]
)

print(
    "Hash length:",
    len(findings["evidence_sha256"])
)

print(
    "Integrity status: VERIFIED"
)


# ================================================================
# RUN TESTS
# ================================================================

print("\n")
print("=" * 78)
print("AUTOMATED VALIDATION")
print("=" * 78)

success = run_tests()


# ================================================================
# FINAL RESULT
# ================================================================

print("\n" + "=" * 78)

if success:

    print(
        "EXPERIMENT 12 COMPLETED SUCCESSFULLY"
    )

    print(
        "All 15 test cases passed."
    )

else:

    print(
        "EXPERIMENT 12 COMPLETED WITH FAILURES."
    )

print("=" * 78)

EXPERIMENT 12
CAPSTONE: END-TO-END INVESTIGATION
OF A SIMULATED INSIDER DATA THEFT


CAPSTONE INVESTIGATION SUMMARY - CASE/CYB/2026/0417
Events examined : 12
Evidence file SHA-256 : 479667f1480ad2c7d5480bfc9e998d9c3c26b21f12f2a6c52d6f773c68cd858a
Distinct users observed : r.kumar, svc_backup, svc_helper
External addresses : 10.20.3.41, 203.0.113.90
External domains : files.example.net
Beaconing detected : True (interval 1800 s)
Large outbound transfers : 1
 20-08-2026 06:01 IST 41,231,872 bytes
Privileged account created : 1
Audit log cleared : 1
Out-of-hours events : 9

INDICATIVE LEGAL PROVISIONS (educational mapping, not legal advice)
Unauthorised access / dishonest use of another account    IT Act s.66 r/w s.43
Fraudulent use of another person's credentials            IT Act s.66C
Disclosure of information in breach of lawful contract    IT Act s.72A
Copying and exfiltrating employer data                    IT Act s.43(b) and s.66
Deletion or alteration of audit records            